✅ Step 1: Load & Inspect Data

In [163]:
# ✅ Step 1: Load & Inspect Data
import pandas as pd
import numpy as np

# Load your dataset
df = pd.read_csv("data/expense_data_2.csv")

# Peek at the top
print("📦 Loaded Dataset:")
display(df.head())

📦 Loaded Dataset:


,Income,Age,Dependents,Occupation,City_Tier,Rent,Loan_Repayment,Insurance,Groceries,Transport,...,Desired_Savings,Disposable_Income,Potential_Savings_Groceries,Potential_Savings_Transport,Potential_Savings_Eating_Out,Potential_Savings_Entertainment,Potential_Savings_Utilities,Potential_Savings_Healthcare,Potential_Savings_Education,Potential_Savings_Miscellaneous
0,44637.249636,49,0,Self_Employed,Tier_1,13391.174891,0.000000,2206.490129,6658.768341,2636.970696,...,6200.537192,11265.627707,1685.696222,328.895281,465.769172,195.151320,678.292859,67.682471,0.000000,85.735517
1,26858.596592,34,2,Retired,Tier_2,5371.719318,0.000000,869.522617,2818.444460,1543.018778,...,1923.176434,9676.818733,540.306561,119.347139,141.866089,234.131168,286.668408,6.603212,56.306874,97.388606
2,50367.605084,35,1,Student,Tier_3,7555.140763,4612.103386,2201.800050,6313.222081,3221.396403,...,7050.360422,13891.450624,1466.073984,473.549752,410.857129,459.965256,488.383423,7.290892,106.653597,138.542422
3,101455.600247,21,0,Self_Employed,Tier_3,15218.340037,6809.441427,4889.418087,14690.149363,7106.130005,...,16694.965136,31617.953615,1875.932770,762.020789,1241.017448,320.190594,1389.815033,193.502754,0.000000,296.041183
4,24875.283548,52,4,Professional,Tier_2,4975.056710,3112.609398,635.907170,3034.329665,1276.155163,...,1874.099434,6265.700532,788.953124,68.160766,61.712505,187.173750,194.117130,47.294591,67.388120,96.557076


✅ Step 2: Preprocess the Data

In [164]:
# ✅ Step 2: Preprocess Data
# Drop unnecessary targets
df = df.drop(columns=["Disposable_Income", "Loan_Repayment"], errors='ignore')

# Fill missing numeric values
numeric_cols = df.select_dtypes(include='number').columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

# Add income transformations
df["Income_Log"] = np.log1p(df["Income"])
df["Income_Squared"] = df["Income"] ** 2


✅ STEP 3: Define Inputs & Outputs

In [165]:
# ✅ Step 3: Define Inputs & Outputs
expense_columns = [
    "Rent", "Insurance", "Groceries", "Transport",
    "Eating_Out", "Entertainment", "Utilities",
    "Healthcare", "Miscellaneous"
]


input_columns = ["Income", "Income_Log", "Income_Squared"] + expense_columns
X = df[input_columns]
y = df[expense_columns]

✅ Step 4: Train-Test Split

In [166]:
# ✅ Step 4: Train-Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


✅ Step 5: Train the Model (MultiOutput XGBoost)

In [167]:
# ✅ Step 5: Train MultiOutput XGBoost Model
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor

xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.4,
    reg_lambda=0.4,
    random_state=42
)

model = MultiOutputRegressor(xgb)
model.fit(X_train, y_train)

MultiOutputRegressor(estimator=XGBRegressor(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample_bynode=None,
                                            colsample_bytree=0.8, device=None,
                                            early_stopping_rounds=None,
                                            enable_categorical=False,
                                            eval_metric=None,
                                            feature_types=None, gamma=None,
                                            grow_policy=None,
                                            importance_type=None,
                                            interaction_constraints=None,
                                            learning_rate=0.03, max_bin=None,
                                            max_cat_threshold=None,
                                            max_cat_to_onehot=None,
                                            max_delta_step=None, max_depth=4,
                                            max_leaves=None,
                                            min_child_weight=None, missing=nan,
                                            monotone_constraints=None,
                                            multi_strategy=None,
                                            n_estimators=300, n_jobs=None,
                                            num_parallel_tree=None,
                                            random_state=42, ...))

✅ Step 6: Evaluate the Model

In [168]:
# ✅ Step 6: Evaluate the Model
from sklearn.metrics import mean_absolute_error, r2_score

y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred, multioutput='raw_values')
mae = mean_absolute_error(y_test, y_pred, multioutput='raw_values')

print("\ud83d\udcca Final Unified Model Evaluation:")
for i, col in enumerate(expense_columns):
    print(f"{col}: R² = {r2[i]:.4f}, MAE = {mae[i]:.2f}")

Exception in callback BaseAsyncIOLoop._handle_events(1124, 1)
handle: <Handle BaseAsyncIOLoop._handle_events(1124, 1)>
Traceback (most recent call last):
  File "C:\Users\Ravindu\AppData\Roaming\Python\Python310\site-packages\jupyter_client\session.py", line 95, in json_packer
    return json.dumps(
UnicodeEncodeError: 'utf-8' codec can't encode characters in position 28-29: surrogates not allowed

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\Users\Ravindu\AppData\Local\Programs\Python\Python310\lib\asyncio\events.py", line 80, in _run
    self._context.run(self._callback, *self._args)
  File "C:\Users\Ravindu\AppData\Roaming\Python\Python310\site-packages\tornado\platform\asyncio.py", line 202, in _handle_events
    handler_func(fileobj, events)
  File "C:\Users\Ravindu\AppData\Roaming\Python\Python310\site-packages\zmq\eventloop\zmqstream.py", line 600, in _handle_events
    self._handle_recv()
  File "C:\Users\Rav

✅ STEP 7: Save the Model

In [169]:
# ✅ Step 7: Save Model
import joblib
joblib.dump(model, "unified_expense_predictor.pkl")
print("\u2705 Model saved as unified_expense_predictor.pkl")

✅ Model saved as unified_expense_predictor.pkl


✅ STEP 8: Smart Prediction Logic (Handles Partial or No Input)

In [170]:
# ✅ Define fallback income for no-input prediction
fallback_income = df["Income"].median()


In [171]:
def predict_expenses_from_any_input(user_input, model, expense_columns):
    import joblib
    income_model = joblib.load("income_predictor.pkl")
    edu_model = joblib.load("education_predictor.pkl")

    main_expenses = [col for col in expense_columns if col != "Education"]
    all_input_cols = ["Income", "Income_Log", "Income_Squared"] + main_expenses

    input_data = {col: [np.nan] for col in all_input_cols}
    for key, value in user_input.items():
        if key in input_data:
            input_data[key] = [value]

    if pd.isna(input_data["Income"][0]):
        known_expenses = [col for col in main_expenses if not pd.isna(input_data[col][0])]
        if known_expenses:
            income_input_df = pd.DataFrame({
                col: [input_data[col][0] if col in known_expenses else 0]
                for col in main_expenses
            })
            predicted_income = income_model.predict(income_input_df)[0]
            input_data["Income"] = [predicted_income]
        else:
            input_data["Income"] = [fallback_income]

    income = input_data["Income"][0]
    input_data["Income_Log"] = [np.log1p(income)]
    input_data["Income_Squared"] = [income ** 2]

    df_input = pd.DataFrame(input_data)
    X_input = df_input[model.feature_names_in_].fillna(0)

    # 🔹 Predict main expenses
    main_pred = model.predict(X_input)[0]
    result = {col: main_pred[i] for i, col in enumerate(main_expenses)}

    # 🔹 Predict Education (log-scale model)
    try:
        X_edu_input = df_input[model.feature_names_in_]  # use same features
        edu_log_pred = edu_model.predict(X_edu_input)[0]
        edu_value = np.expm1(edu_log_pred)
        edu_value = max(0, edu_value)
        edu_value = min(edu_value, 0.1 * income)  # Cap at 10% income if needed
        result["Education"] = float(edu_value)
    except:
        result["Education"] = 0.0

    return result, income


✅ Step 9: Test It Out

In [172]:
# ✅ Step 9: Test Prediction (With Income Provided)

user_input = {
    "Income": 85000
}

predicted_expenses, predicted_income = predict_expenses_from_any_input(user_input, model, expense_columns)

print(f"Predicted Income: ${predicted_income:.2f}")
print("🧾 Predicted Full Expenses:")
# Convert dict to DataFrame first, then transpose
import pandas as pd

predicted_df = pd.DataFrame(predicted_expenses, index=["Predicted Amount"]).T
print(predicted_df)


Predicted Income: $85000.00
🧾 Predicted Full Expenses:
               Predicted Amount
Rent                4135.621582
Insurance            896.608032
Groceries           3077.901611
Transport           1595.004395
Eating_Out           760.869507
Entertainment        698.000000
Utilities            950.764099
Healthcare           894.819458
Miscellaneous        400.601410
Education           8500.000000


✅Testing

In [173]:
import pandas as pd

# Convert dictionary to DataFrame
predicted_df = pd.DataFrame.from_dict(predicted_expenses, orient="index", columns=["Predicted ($)"])
predicted_df.index.name = "Expense Type"
predicted_df = predicted_df.reset_index()

print(f"Predicted Income: ${predicted_income:.2f}")
print("\n📊 Predicted Expense Breakdown:")
display(predicted_df)


Predicted Income: $85000.00

📊 Predicted Expense Breakdown:


,Expense Type,Predicted ($)
0,Rent,4135.621582
1,Insurance,896.608032
2,Groceries,3077.901611
3,Transport,1595.004395
4,Eating_Out,760.869507
5,Entertainment,698.000000
6,Utilities,950.764099
7,Healthcare,894.819458
8,Miscellaneous,400.601410
9,Education,8500.000000


🧩 1. Train an Income Prediction Model

In [174]:
# ✅ Train Income Predictor from Known Expenses
from sklearn.ensemble import RandomForestRegressor

# Income as target
X_income = df[expense_columns]
y_income = df["Income"]

# Split for training
X_inc_train, X_inc_test, y_inc_train, y_inc_test = train_test_split(X_income, y_income, test_size=0.2, random_state=42)

# Train simple model
income_model = RandomForestRegressor(n_estimators=100, random_state=42)
income_model.fit(X_inc_train, y_inc_train)

# Optionally save it
joblib.dump(income_model, "income_predictor.pkl")
print("✅ Income model trained and saved.")


✅ Income model trained and saved.


✨ 2. Update the predict_expenses_from_any_input() Function

In [175]:
def predict_expenses_from_any_input(user_input, model, expense_columns):
    # Load income model
    income_model = joblib.load("income_predictor.pkl")

    # Step 1: Prepare full input structure
    all_input_cols = ["Income", "Income_Log", "Income_Squared"] + expense_columns
    input_data = {col: [np.nan] for col in all_input_cols}

    # Step 2: Fill user-provided values
    for key, value in user_input.items():
        if key in input_data:
            input_data[key] = [value]

    # ✅ Step 2.1: If income is missing and some expenses are given, predict income
    if pd.isna(input_data["Income"][0]):
        known_expenses = [col for col in expense_columns if not pd.isna(input_data[col][0])]
        if known_expenses:  # Use known expenses to predict income
            X_for_income = pd.DataFrame({col: [input_data[col][0] if col in known_expenses else 0] for col in expense_columns})
            predicted_income = income_model.predict(X_for_income)[0]
            input_data["Income"] = [predicted_income]
        else:
            input_data["Income"] = [fallback_income]  # use fallback if nothing is known

    # Step 3: Add engineered features if missing
    if pd.isna(input_data["Income_Log"][0]) and not pd.isna(input_data["Income"][0]):
        input_data["Income_Log"] = np.log1p(input_data["Income"][0])
    if pd.isna(input_data["Income_Squared"][0]) and not pd.isna(input_data["Income"][0]):
        input_data["Income_Squared"] = input_data["Income"][0] ** 2

    df_input = pd.DataFrame(input_data)

    # Step 4: Fill remaining with 0
    X_input = df_input[model.feature_names_in_].fillna(0)

    # Step 5: Predict
    prediction = model.predict(X_input)[0]

    # Step 6: Merge known + predicted
    result = df_input[expense_columns].copy()
    for i, col in enumerate(expense_columns):
        if pd.isna(result.at[0, col]):
            val = prediction[i]
            # 👇 Force Education to stay within a plausible range
            if col == "Education":
                val = max(0, val)  # Prevent negative
                val = min(val, 0.10 * income)  # Cap education to 10% of income (tweakable)
            result.at[0, col] = val


    return result, input_data["Income"][0]


🧪 Example Usage

In [177]:
# ✅ Correct way to format and display predicted output

user_input = {}  # absolutely no input
predicted_expenses, predicted_income = predict_expenses_from_any_input(user_input, model, expense_columns)

# Convert predictions to clean format
predicted_df = predicted_expenses.T.rename(columns={0: "Predicted ($)"})
predicted_df.index.name = "Expense Type"
predicted_df = predicted_df.reset_index()

print(f"Predicted Income: ${predicted_income:.2f}")
print("\n📊 Predicted Expense Breakdown:")
display(predicted_df)


Predicted Income: $30185.38

📊 Predicted Expense Breakdown:


,Expense Type,Predicted ($)
0,Rent,2010.480347
1,Insurance,384.274902
2,Groceries,1375.807129
3,Transport,715.045837
4,Eating_Out,367.720978
5,Entertainment,338.505127
6,Utilities,495.345734
7,Healthcare,424.204285
8,Miscellaneous,181.566055
